# Airport assignment accuracy versus distance

Paper-ready comparison of the LightGBM airport model and the nearest-airport baseline (`no_airports_model=True`). The flight algorithm uses only OpenSky ADS-B data. SFDPS supplies the gold takeoff and landing airports; each matched flight contributes one takeoff and one landing observation. Distance is measured from the first/last OpenSky position to the corresponding gold airport.

In [ ]:
from datetime import date, datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from flights.flights_comparison import df_flights_comparision

TARGET_DATE = date(2026, 3, 1)
ADSB_SOURCE = "opensky"
PDF_PATH = Path("output/pdf/airport_accuracy_vs_gold_distance_opensky.pdf")

MODELS = {
    "LightGBM": False,
    "Nearest airport": True,
}
BIN_EDGES_KM = [0, 1, 2, 5, 10, 20, 50, 100, float("inf")]
BIN_LABELS = ["0-1", "1-2", "2-5", "5-10", "10-20", "20-50", "50-100", "100+"]

In [ ]:
start = datetime.combine(TARGET_DATE, datetime.min.time())
end = start + timedelta(days=1)
same_day = (
    (pl.col("takeoff_time").dt.date() == TARGET_DATE)
    & (pl.col("landing_time").dt.date() == TARGET_DATE)
)

gold = get_sfdps_flights_day(TARGET_DATE).filter(same_day)
comparisons = {}

for model_label, no_airports_model in MODELS.items():
    predicted = get_flights(
        TARGET_DATE,
        adsb_src=ADSB_SOURCE,
        no_airports_model=no_airports_model,
    ).filter(same_day)

    # Evaluate the common ICAO population without consulting another ADS-B source.
    predicted = predicted.filter(
        pl.col("icao").is_in(gold.get_column("icao").unique().implode())
    )
    model_gold = gold.filter(
        pl.col("icao").is_in(predicted.get_column("icao").unique().implode())
    )
    comparisons[model_label] = df_flights_comparision(
        predicted,
        model_gold,
        start,
        end,
        compare_airports=False,
    ).filter(pl.col("match_status") == "both")

{label: comparison.height for label, comparison in comparisons.items()}

In [ ]:
observations = []
for model_label, comparison in comparisons.items():
    observations.extend([
        comparison.select(
            pl.lit(model_label).alias("model"),
            pl.lit("Takeoff").alias("endpoint"),
            pl.col("first_message_distance_to_takeoff_airport_km").alias("distance_km"),
            pl.col("same_takeoff_airport_ident").alias("correct"),
        ),
        comparison.select(
            pl.lit(model_label).alias("model"),
            pl.lit("Landing").alias("endpoint"),
            pl.col("last_message_distance_to_landing_airport_km").alias("distance_km"),
            pl.col("same_landing_airport_ident").alias("correct"),
        ),
    ])

observations = (
    pl.concat(observations)
    .drop_nulls(["distance_km", "correct"])
    .with_columns(
        pl.col("distance_km")
        .cut(BIN_EDGES_KM[1:-1], labels=BIN_LABELS, left_closed=True)
        .alias("distance_bin")
    )
)

# Binomial accuracy with a 95% Wilson interval.
z = 1.96
bin_order = {label: index for index, label in enumerate(BIN_LABELS)}
binned = (
    observations
    .group_by("model", "distance_bin")
    .agg(
        pl.len().alias("n"),
        pl.col("correct").mean().alias("accuracy"),
    )
    .with_columns(
        pl.col("distance_bin").cast(pl.String).replace_strict(bin_order).alias("bin_order")
    )
    .with_columns(
        (1 + z**2 / pl.col("n")).alias("wilson_denom"),
        (pl.col("accuracy") + z**2 / (2 * pl.col("n"))).alias("wilson_center_num"),
        (
            z
            * (
                pl.col("accuracy") * (1 - pl.col("accuracy")) / pl.col("n")
                + z**2 / (4 * pl.col("n") ** 2)
            ).sqrt()
        ).alias("wilson_half_num"),
    )
    .with_columns(
        ((pl.col("wilson_center_num") - pl.col("wilson_half_num")) / pl.col("wilson_denom")).alias("lower"),
        ((pl.col("wilson_center_num") + pl.col("wilson_half_num")) / pl.col("wilson_denom")).alias("upper"),
    )
    .sort("bin_order", "model")
)

binned.select("model", "distance_bin", "n", "accuracy", "lower", "upper")

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(3.35, 2.30), constrained_layout=True)
x = list(range(len(BIN_LABELS)))
styles = {
    "LightGBM": {"color": "#0072B2", "marker": "o", "linestyle": "-"},
    "Nearest airport": {"color": "#D55E00", "marker": "s", "linestyle": "--"},
}

for model_label in MODELS:
    model_bins = binned.filter(pl.col("model") == model_label).sort("bin_order")
    accuracy = model_bins.get_column("accuracy").to_numpy() * 100
    lower = model_bins.get_column("lower").to_numpy() * 100
    upper = model_bins.get_column("upper").to_numpy() * 100
    style = styles[model_label]
    ax.plot(
        x,
        accuracy,
        label=model_label,
        linewidth=1.35,
        markersize=3.5,
        markerfacecolor="white",
        markeredgewidth=0.9,
        **style,
    )
    ax.fill_between(x, lower, upper, color=style["color"], alpha=0.12, linewidth=0)

ax.set_xlabel("OpenSky endpoint distance to gold airport (km)")
ax.set_ylabel("Airport accuracy (%)")
ax.set_xticks(x, BIN_LABELS, rotation=28, ha="right", rotation_mode="anchor")
ax.set_ylim(0, 102)
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis="y", color="0.88", linewidth=0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower left", frameon=False)

PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.02,
    metadata={"Creator": "PlaneQuery airport distance accuracy notebook"},
)
print(f"Saved {PDF_PATH}")
plt.show()

## Comparison across ADS-B sources

The second figure repeats the same evaluation for OpenSky, ADSB.lol, and ADS-B Exchange. Faceting by ADS-B source keeps the LightGBM-versus-nearest-airport comparison readable at two-column paper width.

In [ ]:
ADSB_SOURCES = {
    "opensky": "OpenSky",
    "adsblol": "ADSB.lol",
    "adsbx": "ADS-B Exchange",
}
ALL_SOURCES_PDF_PATH = Path("output/pdf/airport_accuracy_vs_gold_distance_all_adsb_sources.pdf")

all_comparisons = {}
for adsb_source, source_label in ADSB_SOURCES.items():
    for model_label, no_airports_model in MODELS.items():
        predicted = get_flights(
            TARGET_DATE,
            adsb_src=adsb_source,
            no_airports_model=no_airports_model,
        ).filter(same_day)
        predicted = predicted.filter(
            pl.col("icao").is_in(gold.get_column("icao").unique().implode())
        )
        model_gold = gold.filter(
            pl.col("icao").is_in(predicted.get_column("icao").unique().implode())
        )
        all_comparisons[(source_label, model_label)] = df_flights_comparision(
            predicted,
            model_gold,
            start,
            end,
            compare_airports=False,
        ).filter(pl.col("match_status") == "both")

{key: comparison.height for key, comparison in all_comparisons.items()}

In [ ]:
all_observations = []
for (source_label, model_label), comparison in all_comparisons.items():
    all_observations.extend([
        comparison.select(
            pl.lit(source_label).alias("adsb_source"),
            pl.lit(model_label).alias("model"),
            pl.lit("Takeoff").alias("endpoint"),
            pl.col("first_message_distance_to_takeoff_airport_km").alias("distance_km"),
            pl.col("same_takeoff_airport_ident").alias("correct"),
        ),
        comparison.select(
            pl.lit(source_label).alias("adsb_source"),
            pl.lit(model_label).alias("model"),
            pl.lit("Landing").alias("endpoint"),
            pl.col("last_message_distance_to_landing_airport_km").alias("distance_km"),
            pl.col("same_landing_airport_ident").alias("correct"),
        ),
    ])

all_observations = (
    pl.concat(all_observations)
    .drop_nulls(["distance_km", "correct"])
    .with_columns(
        pl.col("distance_km")
        .cut(BIN_EDGES_KM[1:-1], labels=BIN_LABELS, left_closed=True)
        .alias("distance_bin")
    )
)

all_binned = (
    all_observations
    .group_by("adsb_source", "model", "distance_bin")
    .agg(
        pl.len().alias("n"),
        pl.col("correct").mean().alias("accuracy"),
    )
    .with_columns(
        pl.col("distance_bin").cast(pl.String).replace_strict(bin_order).alias("bin_order")
    )
    .with_columns(
        (1 + z**2 / pl.col("n")).alias("wilson_denom"),
        (pl.col("accuracy") + z**2 / (2 * pl.col("n"))).alias("wilson_center_num"),
        (
            z
            * (
                pl.col("accuracy") * (1 - pl.col("accuracy")) / pl.col("n")
                + z**2 / (4 * pl.col("n") ** 2)
            ).sqrt()
        ).alias("wilson_half_num"),
    )
    .with_columns(
        ((pl.col("wilson_center_num") - pl.col("wilson_half_num")) / pl.col("wilson_denom")).alias("lower"),
        ((pl.col("wilson_center_num") + pl.col("wilson_half_num")) / pl.col("wilson_denom")).alias("upper"),
    )
    .sort("adsb_source", "bin_order", "model")
)

all_binned.select("adsb_source", "model", "distance_bin", "n", "accuracy", "lower", "upper")

In [ ]:
fig, axes = plt.subplots(
    1,
    len(ADSB_SOURCES),
    figsize=(7.0, 2.45),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
legend_labels = {
    "LightGBM": "Algorithm (LightGBM)",
    "Nearest airport": "Algorithm (nearest airport)",
}
legend_handles = []

for ax, source_label in zip(axes, ADSB_SOURCES.values()):
    for model_label in MODELS:
        model_bins = (
            all_binned
            .filter(
                (pl.col("adsb_source") == source_label)
                & (pl.col("model") == model_label)
            )
            .sort("bin_order")
        )
        model_x = model_bins.get_column("bin_order").to_numpy()
        accuracy = model_bins.get_column("accuracy").to_numpy() * 100
        lower = model_bins.get_column("lower").to_numpy() * 100
        upper = model_bins.get_column("upper").to_numpy() * 100
        style = styles[model_label]
        line, = ax.plot(
            model_x,
            accuracy,
            label=legend_labels[model_label],
            linewidth=1.35,
            markersize=3.2,
            markerfacecolor="white",
            markeredgewidth=0.9,
            **style,
        )
        ax.fill_between(model_x, lower, upper, color=style["color"], alpha=0.12, linewidth=0)
        if len(legend_handles) < len(MODELS):
            legend_handles.append(line)

    ax.set_title(source_label, fontsize=8.5, pad=3)
    ax.set_xticks(x, BIN_LABELS, rotation=38, ha="right", rotation_mode="anchor")
    ax.set_ylim(0, 102)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.grid(axis="y", color="0.88", linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("Airport accuracy (%)")
fig.supxlabel("ADS-B endpoint distance to gold airport (km)", fontsize=8)
fig.legend(
    legend_handles,
    [handle.get_label() for handle in legend_handles],
    loc="outside upper center",
    ncol=2,
    frameon=False,
)

ALL_SOURCES_PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(
    ALL_SOURCES_PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.02,
    metadata={"Creator": "PlaneQuery airport distance accuracy notebook"},
)
print(f"Saved {ALL_SOURCES_PDF_PATH}")
plt.show()